<a href="https://colab.research.google.com/github/ibrahim0015/Internship/blob/main/Neural%20Networks/%20Rice(with%20transfer%20learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip rice_data.zip


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder

In [9]:
import torchvision.models as models
# Load ResNet-18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

num_classes = 5  # Replace with your number of classes
in_features = model.fc.in_features

# Replace the final fully connected layer
model.fc = nn.Linear(in_features, num_classes)




In [ ]:
# class SimpleCNN(nn.Module):
#     def __init__(self, num_classes=5):
#         super(SimpleCNN, self).__init__()
#         self.conv_block1 = nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size=3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2)
#         )
#         self.conv_block2 = nn.Sequential(
#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2)
#         )
#         self.conv_block3 = nn.Sequential(
#             nn.Conv2d(64, 128, kernel_size=3, padding=1),
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2)
#         )
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(128 * 8 * 8, 256),  # 64 -> 32 -> 16 -> 8 after 3 MaxPools
#             nn.ReLU(),
#             nn.Dropout(0.5),
#             nn.Linear(256, num_classes)
#         )

#     def forward(self, x):
#         x = self.conv_block1(x)
#         x = self.conv_block2(x)
#         x = self.conv_block3(x)
#         x = self.classifier(x)
#         return x

In [10]:
def get_dataloaders(batch_size=32, test_split=0.2):
    train_transform = transforms.Compose([
        transforms.Resize((256)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])



    test_transform = transforms.Compose([
        transforms.Resize((256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = ImageFolder(root="/content/Rice_Image_Dataset")

    total_size = len(full_dataset)
    test_size = int(test_split * total_size)
    train_size = total_size - test_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

    # Apply separate transforms after splitting
    train_dataset = ImageFolder(root="/content/Rice_Image_Dataset", transform=train_transform)
    test_dataset  = ImageFolder(root="/content/Rice_Image_Dataset", transform=test_transform)

    train_dataset, _ = random_split(train_dataset, [train_size, test_size])
    _, test_dataset  = random_split(test_dataset,  [train_size, test_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    print(f"Classes:        {full_dataset.classes}")
    print(f"Total images:   {total_size}")
    print(f"Training set:   {train_size}")
    print(f"Test set:       {test_size}")

    return train_loader, test_loader

In [11]:
def train_model(model, train_loader, criterion, optimizer, scheduler, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total
        avg_loss  = running_loss / len(train_loader)

        scheduler.step()  # Decay LR every step_size epochs

        print(f"Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.4f}  Train Acc: {train_acc:.2f}%")


In [12]:
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_loader, test_loader = get_dataloaders(batch_size=32)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("\nStarting training...")
train_model(model, train_loader, criterion, optimizer, scheduler, epochs=5)

# Unfreeze layer4 (last residual block)
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.00001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

train_model(model, train_loader, criterion, optimizer, scheduler, epochs=15)

print("\nEvaluating on test set...")
evaluate_model(model, test_loader)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Classes:        ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']
Total images:   75000
Training set:   60000
Test set:       15000

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch [1/5]  Loss: 0.4930  Train Acc: 88.63%
Epoch [2/5]  Loss: 0.1877  Train Acc: 95.50%
Epoch [3/5]  Loss: 0.1403  Train Acc: 96.16%
Epoch [4/5]  Loss: 0.1177  Train Acc: 96.55%
Epoch [5/5]  Loss: 0.1048  Train Acc: 96.92%
Epoch [1/15]  Loss: 0.0366  Train Acc: 98.83%
Epoch [2/15]  Loss: 0.0161  Train Acc: 99.50%
Epoch [3/15]  Loss: 0.0122  Train Acc: 99.62%
Epoch [4/15]  Loss: 0.0108  Train Acc: 99.66%


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Hook to capture activation
activation = {}

def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

# Register hook on a valid layer of ResNet-18
# For ResNet-18, layers are typically named conv1, layer1, layer2, etc.
# Let's pick a convolutional layer from 'layer2' block for visualization
model.layer2[0].conv1.register_forward_hook(get_activation('layer2_conv1'))

# Forward pass a single image
model.eval()
images, labels = next(iter(test_loader))   # grab one batch
single_image = images[0].unsqueeze(0).to(device)  # shape: (1, 3, 224, 224) (after transform)

with torch.no_grad():
    output = model(single_image)

# Plot the activation maps
act = activation['layer2_conv1'].squeeze(0)
print(f"Activation map shape: {act.shape}")

# Display a subset of activation maps if there are too many channels
num_channels_to_display = min(act.shape[0], 64) # Display max 64 channels

fig, axes = plt.subplots(int(np.ceil(num_channels_to_display/8)), 8, figsize=(12, int(np.ceil(num_channels_to_display/8))*1.5))
for i, ax in enumerate(axes.flat):
    if i < num_channels_to_display:
        ax.imshow(act[i].cpu(), cmap='viridis')
    ax.axis('off')

plt.suptitle('Activation Maps — layer2_conv1', fontsize=14)
plt.tight_layout()
plt.show()

# original image for comparison
# Define the mean and std used in normalization for correct de-normalization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

original = single_image.squeeze(0).cpu().permute(1, 2, 0).numpy()
original = original * std + mean # De-normalize

# Clip values to [0, 1] in case of slight out-of-range due to floating point arithmetic
original = np.clip(original, 0, 1)

plt.figure(figsize=(3, 3))
plt.imshow(original)
plt.title(f"Original — Label: {labels[0].item()}")
plt.axis('off')
plt.show()